# M-ViSER - Speech Emotion Recognition

**Repo**: https://github.com/Huu2412/M-ViSER

---
### Training Modes
| Stage | Mo ta | Lenh |
|---|---|---|
| **0** | End-to-End (Student + Teacher cung luc) | `--stage 0` |
| **1** | Chi train Teacher (Audio + Clean Text) | `--stage 1` |
| **2** | Student Distillation tu Teacher da freeze | `--stage 2 --teacher_ckpt ...` |

> **Recommended**: Chay Stage 1 truoc -> lay checkpoint -> Stage 2

> **QUAN TRONG**: Chay tung cell theo thu tu 1 -> 2 -> 3 -> 4 -> 5x


In [ ]:
# ============================================================
# CELL 1: Clone repo (luon lay code moi nhat)
# ============================================================
import os

REPO_URL = 'https://github.com/Huu2412/M-ViSER.git'
REPO_DIR = '/kaggle/working/M-ViSER'

os.system(f'rm -rf {REPO_DIR}')
os.system(f'git clone {REPO_URL} {REPO_DIR}')

print('=== Latest 3 commits ===')
os.system(f'git -C {REPO_DIR} log --oneline -3')


In [ ]:
# ============================================================
# CELL 2: Fix CUDA compatibility + Cai dat dependencies
#
# Ly do: neu torch duoc cai de tu PyPI (pip install torch)
# se gay loi: 'CUDA error: no kernel image is available'
# vi ban PyPI khong co CUDA kernel cho GPU cua Kaggle.
#
# Giai phap: Kiem tra CUDA hoat dong, neu khong thi restore
# torch dung version cho Kaggle GPU.
# ============================================================
import subprocess, sys, os

def run_pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip'] + list(args),
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('pip stderr:', r.stderr[-500:])
    return r.returncode

# --- Buoc 1: Kiem tra torch CUDA co hoat dong khong ---
import torch
print(f'Current PyTorch : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

cuda_ok = False
if torch.cuda.is_available():
    try:
        _x = torch.tensor([1.0, float('nan')]).cuda()
        torch.isfinite(_x)  # operation bi loi neu CUDA build sai
        del _x
        cuda_ok = True
        print('CUDA kernel test : OK')
    except Exception as e:
        print(f'CUDA kernel test : FAILED ({e})')
        cuda_ok = False

# --- Buoc 2: Neu CUDA bi hong -> restore torch dung version ---
if not cuda_ok and torch.cuda.is_available():
    print()
    print('==> Phat hien torch CUDA khong tuong thich.')
    print('    Dang restore torch cho Kaggle GPU...')

    # Phat hien CUDA version tu nvidia-smi
    cuda_ver_raw = os.popen('nvidia-smi | grep "CUDA Version" | awk "{print $9}"').read().strip()
    print(f'    nvidia-smi CUDA version: {cuda_ver_raw}')

    # Chon whl index phu hop voi CUDA version
    if cuda_ver_raw.startswith('12'):
        cuda_tag = 'cu121'   # Kaggle 2024+ su dung CUDA 12.x
    elif cuda_ver_raw.startswith('11'):
        cuda_tag = 'cu118'
    else:
        cuda_tag = 'cu121'   # fallback

    whl_url = f'https://download.pytorch.org/whl/{cuda_tag}'
    print(f'    Cai dat tu: {whl_url}')

    rc = run_pip('install', '--force-reinstall', '-q',
                 'torch', 'torchaudio',
                 '--index-url', whl_url)

    # Restart kernel sau khi cai lai torch
    if rc == 0:
        print('    Torch restored! PHAI RESTART KERNEL va chay lai tu Cell 1.')
        print('    (Kaggle: Runtime -> Restart Session -> Run All)')
    else:
        print('    Cai dat that bai! Kiem tra ket noi mang.')

# --- Buoc 3: Cai cac thu vien khac (KHONG cai torch) ---
SKIP_PKGS = {'torch', 'torchaudio', 'torchvision'}

req_path = f'{REPO_DIR}/requirements.txt'
with open(req_path) as f:
    lines = f.readlines()

pkgs_to_install = []
for line in lines:
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    pkg_name = line.split('>=')[0].split('==')[0].split('<=')[0].strip().lower()
    if pkg_name in SKIP_PKGS:
        print(f'[SKIP] {line}  (giu torch cua Kaggle, tranh CUDA mismatch)')
        continue
    pkgs_to_install.append(line)

print(f'\nCai dat {len(pkgs_to_install)} packages...')
run_pip('install', '-q', *pkgs_to_install)
print('Done.')


In [ ]:
# ============================================================
# CELL 3: Kiem tra moi truong GPU chi tiet
# ============================================================
import torch, os

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {vram_gb:.1f} GB')
    print(f'CUDA version    : {torch.version.cuda}')
    print(f'Compute Cap.    : sm_{props.major}{props.minor}')

    # Xac nhan CUDA kernel hoat dong
    try:
        _x = torch.tensor([1.0, float('nan')]).cuda()
        result = torch.isfinite(_x)
        del _x
        print('CUDA kernel test: PASSED')
    except Exception as e:
        print(f'CUDA kernel test: FAILED -> {e}')
        print('Hay quay lai Cell 2 de fix CUDA roi restart kernel!')
else:
    print('WARNING: Khong co GPU! Hay bat GPU trong Settings.')

os.system('df -h /kaggle/working')


In [ ]:
# ============================================================
# CELL 4: Smoke Test (kiem tra forward + backward pass)
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'Working dir: {os.getcwd()}')
ret = os.system('CUDA_LAUNCH_BLOCKING=1 python smoke_test.py')
print('\nSmoke test: PASSED' if ret == 0 else '\nSmoke test: FAILED')


In [ ]:
# ============================================================
# CELL 5A: Train -- End-to-End (Stage 0)
#   Student + Teacher cung hoc song song.
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

os.system('python train.py --config config/config.yaml --stage 0')


In [ ]:
# ============================================================
# CELL 5B: Train -- Stage 1: Teacher Only
#   Checkpoint luu vao: checkpoints/stage1_teacher/best_model.pt
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

os.system('python train.py --config config/config.yaml --stage 1')


In [ ]:
# ============================================================
# CELL 5C: Train -- Stage 2: Student Distillation
#   Can chay Cell 5B truoc de co teacher checkpoint!
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

TEACHER_CKPT = f'{REPO_DIR}/checkpoints/stage1_teacher/best_model.pt'

if not os.path.exists(TEACHER_CKPT):
    print(f'Teacher checkpoint khong tim thay: {TEACHER_CKPT}')
    print('  --> Hay chay Cell 5B (Stage 1) truoc!')
else:
    print(f'Teacher checkpoint: {TEACHER_CKPT}')
    os.system(
        f'python train.py --config config/config.yaml '
        f'--stage 2 --teacher_ckpt {TEACHER_CKPT}'
    )


In [ ]:
# ============================================================
# CELL 6: 5-Fold Cross-Validation
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

FOLDS = '1 2 3 4 5'  # doi thanh '1 2' de thu nhanh
os.system(f'python run_5fold.py --config config/config.yaml --folds {FOLDS}')


In [ ]:
# ============================================================
# CELL 7: Evaluate tren test set
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

BEST_CKPT = f'{REPO_DIR}/checkpoints/best_model.pt'
candidates = [
    f'{REPO_DIR}/checkpoints/stage2_student/best_model.pt',
    f'{REPO_DIR}/checkpoints/stage1_teacher/best_model.pt',
]
for c in candidates:
    if not os.path.exists(BEST_CKPT) and os.path.exists(c):
        BEST_CKPT = c

print(f'Evaluating checkpoint: {BEST_CKPT}')
os.system(f'python evaluate.py --config config/config.yaml --checkpoint {BEST_CKPT}')


In [ ]:
# ============================================================
# CELL 8: Nen va export checkpoint (tai ve tu Kaggle Output tab)
# ============================================================
import os, shutil
from datetime import datetime

OUTPUT_DIR = '/kaggle/working'
CKPT_DIR   = f'{REPO_DIR}/checkpoints'
ts         = datetime.now().strftime('%Y%m%d_%H%M')
zip_base   = f'{OUTPUT_DIR}/mvisar_ckpt_{ts}'

if os.path.exists(CKPT_DIR):
    shutil.make_archive(zip_base, 'zip', CKPT_DIR)
    zip_file = zip_base + '.zip'
    size_mb  = os.path.getsize(zip_file) / 1e6
    print(f'Checkpoints nen xong: {zip_file}')
    print(f'Kich thuoc: {size_mb:.1f} MB')
    print('Tai ve tu Kaggle > Output tab')
else:
    print(f'Khong tim thay checkpoints: {CKPT_DIR}')

print('\n=== Files trong Kaggle Output ===')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {fname}: {size_mb:.1f} MB')
